# 📘 NumPy 실전 응용

지금까지 배운 NumPy 기술을 실제 데이터 분석 문제에 적용해봅니다.

**학습 목표:**
- 데이터 정규화와 표준화
- 불리언 마스킹과 조건부 연산
- 파일 입출력 (CSV, 텍스트)
- 실전 데이터 분석 예제

## 1. 데이터 정규화와 표준화

머신러닝과 통계 분석에서 데이터의 스케일을 맞추는 것은 매우 중요합니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  정규화(Min-Max)와 표준화(Z-score)       │
# └─────────────────────────────────────────┘

# 학생 점수 데이터
rng = np.random.default_rng(42)
korean = rng.integers(40, 100, size=20)
math = rng.integers(30, 95, size=20)
english = rng.integers(50, 100, size=20)

print("=== 원본 점수 (0~100) ===")
print(f"국어: 평균={korean.mean():.1f}, 표준편차={korean.std():.1f}")
print(f"수학: 평균={math.mean():.1f}, 표준편차={math.std():.1f}")
print(f"영어: 평균={english.mean():.1f}, 표준편차={english.std():.1f}")

# Min-Max 정규화 (0~1 범위)
def normalize(arr):
    return (arr - arr.min()) / (arr.max() - arr.min())

korean_norm = normalize(korean)
math_norm = normalize(math)
english_norm = normalize(english)

print(f"\n=== Min-Max 정규화 (0~1) ===")
print(f"국어: min={korean_norm.min():.3f}, max={korean_norm.max():.3f}")
print(f"수학: min={math_norm.min():.3f}, max={math_norm.max():.3f}")
print(f"영어: min={english_norm.min():.3f}, max={english_norm.max():.3f}")

# Z-score 표준화 (평균=0, 표준편차=1)
def standardize(arr):
    return (arr - arr.mean()) / arr.std()

korean_z = standardize(korean)
math_z = standardize(math)
english_z = standardize(english)

print(f"\n=== Z-score 표준화 (μ=0, σ=1) ===")
print(f"국어: 평균={korean_z.mean():.6f}, 표준편차={korean_z.std():.6f}")
print(f"수학: 평균={math_z.mean():.6f}, 표준편차={math_z.std():.6f}")
print(f"영어: 평균={english_z.mean():.6f}, 표준편차={english_z.std():.6f}")

## 2. 불리언 마스킹과 조건부 연산

조건에 맞는 데이터를 선택하고 처리하는 기법입니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  불리언 마스킹 — 조건으로 데이터 선택      │
# └─────────────────────────────────────────┘

# 50명 학생 점수 데이터
rng = np.random.default_rng(123)
scores = rng.integers(0, 101, size=50)

print(f"학생 수: {len(scores)}")
print(f"평균: {scores.mean():.1f}")
print(f"표준편차: {scores.std():.1f}")

# 불리언 마스킹
pass_mask = scores >= 60
print(f"\n=== 합격/불합격 ===")
print(f"합격자 수: {np.sum(pass_mask)}명")
print(f"불합격자 수: {np.sum(~pass_mask)}명")
print(f"합격률: {np.mean(pass_mask)*100:.1f}%")

# 조건부 통계
print(f"\n=== 합격자 통계 ===")
print(f"합격자 평균: {scores[pass_mask].mean():.1f}")
print(f"불합격자 평균: {scores[~pass_mask].mean():.1f}")

# np.select로 다중 조건
grade = np.select(
    [scores >= 90, scores >= 80, scores >= 70, scores >= 60, scores < 60],
    ['A', 'B', 'C', 'D', 'F']
)
unique, counts = np.unique(grade, return_counts=True)
print(f"\n=== 학점 분포 ===")
for g, c in zip(unique, counts):
    print(f"  {g}: {c}명 ({c/len(scores)*100:.0f}%)")

# np.clip으로 값 제한
clipped = np.clip(scores, 0, 100)  # 이미 0~100이지만 예시
print(f"\nnp.clip(50~80 범위 제한): {np.clip(scores, 50, 80)[:10]}...")

## 3. 파일 입출력

NumPy 배열을 파일로 저장하고 불러오는 방법입니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  NumPy 파일 입출력                       │
# │  .npy, .npz, .txt 형식 지원              │
# └─────────────────────────────────────────┘

import tempfile
import os

# 임시 디렉토리 생성
tmpdir = tempfile.mkdtemp()

# 데이터 준비
data = np.random.default_rng(42).normal(loc=50, scale=10, size=(100, 3))
print(f"데이터 shape: {data.shape}")

# 1. .npy 형식 (NumPy 전용, 빠름)
np.save(os.path.join(tmpdir, 'data.npy'), data)
loaded = np.load(os.path.join(tmpdir, 'data.npy'))
print(f"\n.npy 로드: {loaded.shape}")

# 2. .npz 형식 (여러 배열 저장)
scores = rng.integers(0, 101, size=50)
np.savez(os.path.join(tmpdir, 'multi.npz'),
         data=data, scores=scores)
multi = np.load(os.path.join(tmpdir, 'multi.npz'))
print(f"\n.npz 파일의 배열 목록: {list(multi.keys())}")
print(f"  data: {multi['data'].shape}")
print(f"  scores: {multi['scores'].shape}")

# 3. 텍스트 파일 (CSV 형식)
np.savetxt(os.path.join(tmpdir, 'data.csv'), data[:5],
           delimiter=',', header='col1,col2,col3', comments='')
loaded_csv = np.loadtxt(os.path.join(tmpdir, 'data.csv'), delimiter=',', skiprows=1)
print(f"\nCSV 로드: {loaded_csv.shape}")
print(f"처음 3행:\n{loaded_csv[:3].round(2)}")

# 정리
import shutil
shutil.rmtree(tmpdir)
print("\n임시 파일 정리 완료")

## 4. 실전 예제 — 성적 분석

종합적인 NumPy 기술을 활용한 데이터 분석 예제입니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  실전 예제 — 학급 성적 분석              │
# │  지금까지 배운 모든 기술 종합 활용        │
# └─────────────────────────────────────────┘

rng = np.random.default_rng(2024)

# 30명 학생의 5과목 성적 데이터 생성
subjects = ['국어', '수학', '영어', '과학', '사회']
scores = np.column_stack([
    rng.integers(40, 100, size=30),  # 국어
    rng.integers(30, 100, size=30),  # 수학
    rng.integers(50, 100, size=30),  # 영어
    rng.integers(35, 100, size=30),  # 과학
    rng.integers(45, 100, size=30),  # 사회
])

print("=== 학급 성적 분석 ===")
print(f"학생 수: {scores.shape[0]}명, 과목 수: {scores.shape[1]}개")

# 과목별 통계
print(f"\n{'과목':>6s} {'평균':>6s} {'표준편차':>8s} {'최고':>6s} {'최저':>6s} {'합격률':>6s}")
print("-" * 40)
for i, subj in enumerate(subjects):
    col = scores[:, i]
    pass_rate = np.mean(col >= 60) * 100
    print(f"{subj:>6s} {col.mean():>6.1f} {col.std():>8.1f} {col.max():>6d} {col.min():>6d} {pass_rate:>5.0f}%")

# 학생별 총점과 평균
total = scores.sum(axis=1)
avg = scores.mean(axis=1)
print(f"\n=== 학생별 총점/평균 (상위 5명) ===")
top5_idx = np.argsort(total)[::-1][:5]
for rank, idx in enumerate(top5_idx, 1):
    print(f"{rank}등: 총점={total[idx]:>3d}, 평균={avg[idx]:>5.1f}")

# 상관관계 (과목 간)
print(f"\n=== 과목 간 상관관계 ===")
corr = np.corrcoef(scores.T)
for i, s1 in enumerate(subjects):
    for j, s2 in enumerate(subjects):
        if i < j and abs(corr[i, j]) > 0.3:
            print(f"  {s1}-{s2}: {corr[i, j]:.3f}")

## 🎯 연습 문제

1. 정규분포 N(μ=170, σ=10)에서 1000명의 키 데이터를 생성하고, 180cm 이상인 사람의 비율을 구하세요.
2. 3과목 성적 데이터를 Z-score로 표준화한 후, 표준화 전후의 평균과 표준편차를 비교하세요.
3. `np.savetxt()`로 배열을 저장하고, 다시 로드하여 동일한지 확인하세요.
4. 1000개의 정규분포 난수를 생성하고, 사분위수(Q1, Q2, Q3)를 계산하세요.
5. 5×5 마방진(가로/세로/대각선 합이 같음)이 맞는지 NumPy로 검증하세요.